# InternVL main study

Continue the reviewed VisDSR experiment with **OpenGVLab/InternVL3_5-8B-HF**,
revision `741a7d03020411e666c6109218ab71e08151ef86`. Qwen's 400 main responses
are complete. Both 60-response calibrations have been reviewed.

The notebook uses the frozen release at commit
`2beb395e176de5527ec3d0166529b51a4f1c17e3`. The existing tasks, original
images, five conditions, prompts, JSON schema, scoring and generation settings
are preserved. It calls only InternVL for new inference and never retries an
answer for poor accuracy or formatting. Weights run locally on Kaggle's free GPU.

## Before starting

1. Import this notebook into a **new Kaggle notebook**.
2. Attach **visdsr_qwen_main_complete.zip** as a private Dataset. Attach only one
   VisDSR backup. On a later resume, attach the newest combined results export.
3. Enable **GPU T4 x2** and **Internet**, and check your remaining free GPU quota.
4. Run the two code cells in order. For a background run, use
   **Save Version > Save & Run All**. That starts a fresh session, so the backup
   must be attached as a Dataset before starting.
5. The setup output should show **Qwen main 400/400; both calibrations 60/60**
   and the number of InternVL main responses restored. A new run starts at 0/400.

InternVL calibration averaged **92.20 seconds per call**. That gives a rough
estimate of **2.56 hours per 100 calls**, or **10.24 hours for 400**, excluding
setup, model loading and runtime variation. This is not a completion guarantee.
[Kaggle documents](https://www.kaggle.com/docs/notebooks) a 12-hour limit for
CPU/GPU saved runs. The notebook exports after each batch of at most 100 new
calls, retains numbered checkpoint ZIPs, and resumes from cached responses.

The input validation must pass before the model loads. If it reports a pixel,
checksum, or freeze mismatch, preserve the backup and share the error. Do not
regenerate the images or edit the frozen protocol to get past the check.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

project = Path("/kaggle/working/VisDSR_v2")
frozen_commit = "2beb395e176de5527ec3d0166529b51a4f1c17e3"
if not project.parent.exists():
    raise RuntimeError(
        "Import this notebook into Kaggle with free GPU and Internet enabled."
    )
if not (project / "FREEZE.json").is_file():
    if project.exists():
        raise RuntimeError(
            "The project folder is incomplete. Start a fresh Kaggle session."
        )
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            "v2.0-frozen",
            "https://github.com/Abhishekgit01/VisDSR.git",
            str(project),
        ],
        check=True,
    )
revision = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=project,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if revision != frozen_commit:
    raise RuntimeError(
        "Use a fresh session: this notebook requires the reviewed frozen commit."
    )
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(project / "environment/requirements.txt"),
    ],
    cwd=project,
    check=True,
)
if not list((project / "results/cache").glob("*.json")):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "study_transfer",
            "--restore-auto",
            "/kaggle/input",
            "--require-backup",
        ],
        cwd=project,
        check=True,
    )
saved = {}
for path in (project / "results/cache").glob("*.json"):
    record = json.loads(path.read_text())
    group = (record["split"], record["model_id"])
    pair = (record["task_id"], record["condition"])
    pairs = saved.setdefault(group, set())
    if pair in pairs:
        raise RuntimeError("Duplicate task/condition response in the restored cache.")
    pairs.add(pair)
qwen = "Qwen/Qwen3-VL-8B-Instruct"
internvl = "OpenGVLab/InternVL3_5-8B-HF"
qwen_main = len(saved.get(("main", qwen), set()))
internvl_main = len(saved.get(("main", internvl), set()))
if qwen_main != 400:
    raise RuntimeError(
        f"Restored {qwen_main}/400 Qwen main responses. Attach the verified "
        "visdsr_qwen_main_complete.zip backup as the only VisDSR input."
    )
for model_id in (qwen, internvl):
    if len(saved.get(("pilot2", model_id), set())) != 60:
        raise RuntimeError(f"The reviewed calibration is incomplete for {model_id}.")
if not 0 <= internvl_main <= 400:
    raise RuntimeError("Unexpected InternVL main response count.")
print(f"RESTORED: Qwen main {qwen_main}/400; both calibrations 60/60.", flush=True)
print(
    f"INTERNVL MAIN: {internvl_main}/400 saved; {400 - internvl_main} remain.",
    flush=True,
)
print(
    "The next cell verifies the freeze and input images before InternVL inference.",
    flush=True,
)

## Run the InternVL main calls

Exports a backup after each batch and after a normal interruption.


In [ ]:
import csv
import shutil
import subprocess
import sys
from pathlib import Path

project = Path("/kaggle/working/VisDSR_v2")
latest = Path("/kaggle/working/visdsr_v2_results.zip")
if not (project / "FREEZE.json").is_file():
    raise RuntimeError(
        "Run the setup cell first, or use Save Version > Save & Run All."
    )
try:
    for batch in range(1, 5):
        print(f"INTERNVL BATCH {batch}/4: up to 100 new calls", flush=True)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "notebooks.kaggle_main",
                "--root",
                str(project),
                "--model",
                "model2",
                "--max-new-calls",
                "100",
                "--export",
                str(latest),
            ],
            cwd=project,
            check=True,
        )
        with (project / "results/scores_model2.csv").open(newline="") as source:
            completed = len(list(csv.DictReader(source)))
        checkpoint = latest.with_name(f"visdsr_internvl_main_{completed:03d}.zip")
        shutil.copy2(latest, checkpoint)
        print(f"CHECKPOINT: {completed}/400; {checkpoint.name}", flush=True)
        if completed == 400:
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "analysis.analyze",
                    "--split",
                    "main",
                    "--models",
                    "model1",
                    "model2",
                ],
                cwd=project,
                check=True,
            )
            print(
                "MAIN COLLECTION COMPLETE: 400 Qwen + 400 InternVL responses.",
                flush=True,
            )
            break
    else:
        raise RuntimeError(
            "InternVL has fewer than 400 responses. Download the newest backup and resume."
        )
finally:
    subprocess.run([sys.executable, "-m", "manifest"], cwd=project, check=True)
    subprocess.run(
        [sys.executable, "-m", "study_transfer", "--export", str(latest)],
        cwd=project,
        check=True,
    )
    print(
        "BACKUP READY: /kaggle/working/visdsr_v2_results.zip; download it from Output.",
        flush=True,
    )

## Download the result

Download **visdsr_v2_results.zip** from the saved version's **Output** when the
run finishes. It contains the completed Qwen results, the InternVL progress,
both calibrations, original inputs, raw caches, and score tables. Send the ZIP
for a final audit.

Every completed response is cached immediately, and each completed batch exports
a full backup. A normal interruption also exports. A hard timeout can prevent the
final export; retain the newest available checkpoint and output/cache files.
Only files saved outside the temporary session should be treated as durable.

After **400/400 InternVL**, the notebook computes the combined analysis.
Collection is then complete; the final verification, interpretation, report,
and reproducibility release still need review.
